expanding/exploring 2.1 stuff : )
also testing out mlflow for tracking because its getting hectic

In [1]:
import pandas as pd
import numpy as np
from factor_analyzer import FactorAnalyzer
from src.features import feature_encoder, raw_data_loader
import os 

In [2]:
os.environ["MLFLOW_EXPERIMENT_NAME"] = "2.1-jp-feature-engineering"

In [3]:
df = raw_data_loader.load_and_clean_raw("../")

df = feature_encoder.encode_features(df)

In [4]:
df.head()

,age,bmi,sex,sw_9am_start_diff,sw_5pm_end_diff,nasal_congestion_stuffiness_nose,nasal_blockage_obstr_nose,troub_brth_nose,troub_slp_nose,not_enough_air_excercise_nose,...,pulmonary_problem_other_mdhx,chronic_obstructive_pulmonary_disease_mdhx,asthma_mdhx,cardiovascular_problem_other_mdhx,congestive_heart_failure_mdhx,hypertension_mdhx,oophorectomy_bilateral_mdhx,ahi,dream_recall_frequency_infrequent,dream_recall_frequency_rarely_or_never
0,58.0,30.7,1.0,-1.00,0.0,1.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.0,1.0
1,30.0,29.4,1.0,-1.00,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,1.0,0.0
2,30.0,25.8,1.0,-2.00,2.0,0.0,0.0,0.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,42.0,26.8,0.0,-3.00,3.0,1.0,0.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.9,0.0,1.0
4,36.0,45.2,0.0,-0.75,0.5,3.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.3,0.0,1.0


In [5]:
import mlflow

mlflow.set_tracking_uri("file:///Users/jack/Repos/apnea-predictor/mlruns")
mlflow.set_experiment(os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment"))

/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///Users/jack/Repos/apnea-predictor/mlruns/620249075279939753', creation_time=1776455710940, experiment_id='620249075279939753', last_update_time=1776455710940, lifecycle_stage='active', name='2.1-jp-feature-engineering', tags={}, workspace='default'>

In [6]:
from src.features.transformers import Factor_Analyzer_Transformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from src.utils.data_utils import convert_ahi
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from mlflow.models.signature import infer_signature


X = df.drop(columns=["ahi"])
y = convert_ahi(df["ahi"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_data = pd.concat([X_train, y_train], axis=1)

train_dataset = mlflow.data.from_pandas(
    train_data, name="2.1-sleep-apnea-train-raw", targets="ahi"
)

test_dataset = mlflow.data.from_pandas(
    pd.concat([X_test, y_test], axis=1), name="2.1-sleep-apnea-test-raw", targets="ahi"
)

with mlflow.start_run():
    mlflow.log_input(train_dataset, context="train")
    mlflow.log_input(test_dataset, context="test")
    params = {
        "objective": "binary:logistic",
        "max_depth": 6,
        "learning_rate": 0.1,
        "n_estimators": 100,
        "random_state": 42,
        "test_size": 0.2,
    }
    mlflow.log_params(params)
    model = XGBClassifier(**params)
    model = model.fit(X_train, y_train)
    sig = infer_signature(X_train, model.predict(X_train))

    mlflow.set_tag("model_type", "xgboost")
    mlflow.xgboost.log_model(
        xgb_model=model, name="xgb_model", model_format="json", signature=sig
    )

    y_pred = model.predict(X_test)
    mlflow.log_metric("test_accuracy", np.mean(y_pred == y_test))
    mlflow.log_metric("test_auc", roc_auc_score(y_test, y_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_pred))
    mlflow.log_text(classification_report(y_test, y_pred), "classification_report.txt")

/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [16:19:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "test_size" } 

In [7]:
from src.eval import evaluate_model
fa_transformer = Factor_Analyzer_Transformer(n_factors=18, rotation="promax")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

fa_transformer.fit(X_train)

X_fa = fa_transformer.transform(X)

df_fa = X_fa.copy()
df_fa["ahi"] = y.values

X_train_fa = fa_transformer.transform(X_train)
X_test_fa = fa_transformer.transform(X_test)

train_dataset_fa = mlflow.data.from_pandas(
    pd.concat([X_train_fa, y_train], axis=1), name="2.2-sleep-apnea-train-fa-promax", targets="ahi"
)

test_dataset_fa = mlflow.data.from_pandas(
    pd.concat([X_test_fa, y_test], axis=1), name="2.2-sleep-apnea-test-fa-promax", targets="ahi"
)


with mlflow.start_run():
    mlflow.log_input(train_dataset_fa, context="train")
    mlflow.log_input(test_dataset_fa, context="test")
    params = {
        "objective": "binary:logistic",
        "max_depth": 6,
        "learning_rate": 0.1,
        "n_estimators": 100,
        "random_state": 42,
        "test_size": 0.2,
    }
    mlflow.log_params(params)
    model = XGBClassifier(**params)
    model = model.fit(X_train_fa, y_train)
    sig = infer_signature(X_train_fa, model.predict(X_train_fa))

    mlflow.set_tag("model_type", "xgboost")
    mlflow.xgboost.log_model(
        xgb_model=model, name="xgb_model", model_format="json", signature=sig
    )
    y_pred = model.predict(X_test_fa)
    mlflow.log_metric("test_accuracy", np.mean(y_pred == y_test))
    mlflow.log_metric("test_auc", roc_auc_score(y_test, y_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_pred))
    mlflow.log_text(classification_report(y_test, y_pred), "classification_report.txt")

/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [16:19:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "test_size" } 

In [8]:
fa_transformer.get_top_loadings()

Top loadings for Factor_1:
fati_interferes_with_phys_function_fss       1.031793
fati_prevents_sustained_phys_func_fss        1.019358
fati_interferes_with_responsibilities_fss    0.977720
fati_top_three_disabling_sympts_fss          0.957607
fati_causes_freq_probs_for_me_fss            0.928218
i_am_easily_fati_fss                         0.924449
fati_interferes_with_work_fam_social_fss     0.858591
lower_motivation_when_fati_fss               0.644888
Name: Factor_1, dtype: float64


Top loadings for Factor_2:
diff_staying_asleep_isi                      0.930820
isi_total_score                              0.929114
troub_slp_nose                               0.810117
how_satisfied_with_curr_sleep_pattern_isi    0.794006
slp_quality_sw                               0.713055
diff_falling_asleep_isi                      0.672223
diff_falling_asleep_isq                      0.633961
frequent_wakenings_map                       0.632612
Name: Factor_2, dtype: float64


Top loadings for

In [9]:
df_fa_alt = df.copy()

# Drop columns: 'ess_total_score', 'isi_total_score'
df_fa_alt = df_fa_alt.drop(columns=['ess_total_score', 'isi_total_score'])

In [10]:
def create_versioned_dataset(data, version, base_name="sleep-apnea-data", tags= None):
    """Create a versioned dataset with metadata."""

    dataset = mlflow.data.from_pandas(
        data,
        source=f"data_pipeline_v{version}",
        name=f"{base_name}-v{version}",
        targets="ahi",
    )

    with mlflow.start_run(run_name=f"Dataset_Version_{version}"):
        mlflow.log_inputs([dataset], contexts=["versioning"], tags_list=[tags])

        # Log version metadata
        mlflow.log_params({
            "dataset_version": version,
            "data_size": data.shape,
            "features_count": data.shape[1] - 1,
            "target_distribution": data["ahi"].value_counts().to_dict(),
        })

        # Log data quality metrics
        mlflow.log_metrics({
            "missing_values_pct": (data.isnull().sum().sum() / data.size) * 100,
            "duplicate_rows": data.duplicated().sum(),
            "target_balance": data["ahi"].std(),
        })

    return dataset


In [11]:
"""
varimax fa 
"""
fa_transformer_var = Factor_Analyzer_Transformer(n_factors=18, rotation="varimax")

fa_transformer_var.fit(X_train)

X = fa_transformer_var.transform(X)

df_fa_var = X.copy()
df_fa_var["ahi"] = y.values


In [12]:
fa_transformer_alt = Factor_Analyzer_Transformer(n_factors=18, rotation="promax")

X_alt = df_fa_alt.drop(columns=["ahi"])
y_alt = convert_ahi(df_fa_alt["ahi"])

X_train_alt, X_test_alt, y_train_alt, y_test_alt = train_test_split(
    X_alt, y_alt, test_size=0.2, random_state=42
)

fa_transformer_alt.fit(X_train_alt)
X_train_fa_alt = fa_transformer_alt.transform(X_train_alt)
X_test_fa_alt = fa_transformer_alt.transform(X_test_alt)

X_alt = fa_transformer_alt.transform(X_alt)

df_fa_alt = X_alt.copy()
df_fa_alt["ahi"] = y_alt.values

In [13]:
""" 
key:
alt = dropping ess and isi before factor analysis
var = varimax rotation instead of promax
v1 = raw data
v2 = factor analysis (default promax)
"""

fa_transformer_var_alt = Factor_Analyzer_Transformer(n_factors=18, rotation="varimax")

df_fa_var_alt = df.copy()
df_fa_var_alt.drop(columns=['ess_total_score', 'isi_total_score'], inplace=True)

X_var_alt = df_fa_var_alt.drop(columns=["ahi"])
y_var_alt = df_fa_var_alt["ahi"]

X_train_var_alt, X_test_var_alt, y_train_var_alt, y_test_var_alt = train_test_split(
    X_var_alt, y_var_alt, test_size=0.2, random_state=42
)

fa_transformer_var_alt.fit(X_train_var_alt)

X_var_alt = fa_transformer_var_alt.transform(X_var_alt)

df_fa_var_alt = X_var_alt.copy()
df_fa_var_alt["ahi"] = y_var_alt.values

In [14]:
""" 
putting the 4 variations of the dataset into mlflow with metadata and tags for versioning
"""

tags_default = {"initial_version": "raw_data"}
dataset_v1 = create_versioned_dataset(df, version=1, tags=tags_default)

v2_tags = {"initial_version": "raw_data", "modification": "promax factor analysis features"}
dataset_v2 = create_versioned_dataset(df_fa, version=2, tags=v2_tags)

v2_alt_tags = {"initial_version": "raw_data", "modification": "promax factor analysis features with ess and isi dropped"}
dataset_v2_alt = create_versioned_dataset(df_fa_alt, version=2.1, tags=v2_alt_tags)

v2_var_tags = {"initial_version": "raw_data", "modification": "varimax factor analysis features"}
dataset_v2_var = create_versioned_dataset(df_fa_var, version=2.2, tags=v2_var_tags)

v2_alt_var_tags = {"initial_version": "raw_data", "modification": "varimax factor analysis features with ess and isi dropped"}
dataset_v2_alt_var = create_versioned_dataset(df_fa_var_alt, version=2.3, tags=v2_alt_var_tags)

/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Inte

In [15]:
mlflow.set_experiment("Hyperparameter Tuning Test Experiment")

<Experiment: artifact_location='file:///Users/jack/Repos/apnea-predictor/mlruns/476641595097987911', creation_time=1776467218483, experiment_id='476641595097987911', last_update_time=1776467218483, lifecycle_stage='active', name='Hyperparameter Tuning Test Experiment', tags={}, workspace='default'>

In [21]:
import mlflow
import optuna
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Load data

data = dataset_v2_alt_var.df
X = data.drop(columns=["ahi"])
y = convert_ahi(data["ahi"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.xgboost.autolog(log_models=False)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }

    with mlflow.start_run(nested=True):
        model = XGBClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        return score


with mlflow.start_run():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=50)

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_score", study.best_value)

[I 2026-04-17 16:24:21,619] A new study created in memory with name: no-name-6b29304b-d25c-4425-b9b9-573e592c3b3c
[I 2026-04-17 16:24:22,524] Trial 0 finished with value: 0.5562130177514792 and parameters: {'n_estimators': 261, 'max_depth': 10, 'learning_rate': 0.10528307583365805, 'subsample': 0.61168201691108, 'colsample_bytree': 0.8892409448520993}. Best is trial 0 with value: 0.5562130177514792.
[I 2026-04-17 16:24:22,883] Trial 1 finished with value: 0.5562130177514792 and parameters: {'n_estimators': 117, 'max_depth': 8, 'learning_rate': 0.28373982858316965, 'subsample': 0.8891251775527983, 'colsample_bytree': 0.9685588660645094}. Best is trial 0 with value: 0.5562130177514792.
[I 2026-04-17 16:24:23,606] Trial 2 finished with value: 0.5532544378698225 and parameters: {'n_estimators': 211, 'max_depth': 10, 'learning_rate': 0.11216067411363566, 'subsample': 0.7961973600651958, 'colsample_bytree': 0.821407909941758}. Best is trial 0 with value: 0.5562130177514792.
[I 2026-04-17 16:

notes so far:
try smote and see how that works 
see if you can hyperparam the factor analyzer
look into more factor analyzer things
look at shap 


In [22]:
mlflow.set_experiment("Shapely Test Experiment")

2026/04/17 16:29:20 INFO mlflow.tracking.fluent: Experiment with name 'Shapely Test Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/jack/Repos/apnea-predictor/mlruns/634907438573417250', creation_time=1776468560413, experiment_id='634907438573417250', last_update_time=1776468560413, lifecycle_stage='active', name='Shapely Test Experiment', tags={}, workspace='default'>

In [ ]:
import mlflow
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from mlflow.models import infer_signature


data = dataset_v2_alt.df
X = data.drop(columns=["ahi"])
y = data["ahi"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = xgb.XGBClassifier().fit(X_train, y_train)

# Create evaluation dataset
eval_data = X_test.copy()
eval_data["ahi"] = y_test


with mlflow.start_run():
    signature = infer_signature(X_test, model.predict(X_test))
    model_info = mlflow.sklearn.log_model(model, name="model", signature=signature)

    # Evaluate with SHAP enabled
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data,
        targets="ahi",
        model_type="classifier",
        evaluator_config={"log_explainer": True},
    )

    active_run = mlflow.active_run()
    if active_run is not None:
        print(f"Run ID: {active_run.info.run_id}")
        print(f"Artifact root: {active_run.info.artifact_uri}")

    shap_artifacts = {
        artifact_name: artifact_path
        for artifact_name, artifact_path in result.artifacts.items()
        if "shap" in artifact_name.lower()
    }

    if shap_artifacts:
        print("SHAP artifact paths:")
        for artifact_name, artifact_path in shap_artifacts.items():
            print(f"{artifact_name}: {artifact_path}")
    else:
        print("No SHAP artifacts were found in result.artifacts.")

2026/04/17 16:37:41 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'b275f73fe34549b289f68f719b595528', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow
2026/04/17 16:37:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/17 16:37:42 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-8b5aabba83d14052a0b2cba803f1783b
2026/04/17 16:37:42 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages

Generated: shap_beeswarm_plot
Generated: shap_summary_plot
Generated: shap_feature_importance_plot


## going back to experimenting with the dataset 

In [52]:
from factor_analyzer import Rotator
from sklearn.metrics import classification_report, accuracy_score, recall_score
df = df.copy()

mlflow.xgboost.autolog()

X = df.drop(columns=["ahi"])
y = convert_ahi(df["ahi"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

fa_new = Factor_Analyzer_Transformer(n_factors=18, rotation="oblimax")
fa_new.fit(X_train)
loadings = fa_new.get_top_loadings()

X_train_fa_new = fa_new.transform(X_train)
X_test_fa_new = fa_new.transform(X_test)

oblimax_model = xgb.XGBClassifier().fit(X_train_fa_new, y_train)

eval_data = X_test_fa_new.copy()
eval_data["ahi"] = y_test

preds = oblimax_model.predict(X_test_fa_new)
classification_rep = classification_report(y_test, preds)

acc = accuracy_score(y_test, preds)
recall = recall_score(y_test, preds)




2026/04/20 14:53:41 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'feee6cb949db493e8f8ff17e68f206ef', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow


Top loadings for Factor_1:
fati_causes_freq_probs_for_me_fss                                0.813230
fati_interferes_with_work_fam_social_fss                         0.812796
difficulty_concentrating_fosq                                    0.775423
isi_total_score                                                  0.772655
fati_interferes_with_responsibilities_fss                        0.768649
how_noticeable_is_sleep_problem_impacting_quality_of_life_isi    0.762035
feel_no_energy_freq_phq                                          0.756874
difficulty_remembering_fosq                                      0.751859
Name: Factor_1, dtype: float64


Top loadings for Factor_2:
diff_staying_asleep_isi             0.565074
diff_falling_asleep_isi             0.545368
troub_slp_nose                      0.510390
diff_falling_asleep_isq             0.478709
ess_total_score                     0.469464
frequent_wakenings_map              0.463431
difficulty_staying_asleep_isq       0.451514
proble

2026/04/20 14:53:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [ ]:
df = df.copy()

X = df.drop(columns=["ahi"])
y = convert_ahi(df["ahi"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

oblimax_pipeline = Pipeline([
    ("factor_analysis", Factor_Analyzer_Transformer(n_factors=18, rotation="oblimax")),
    ("xgb_classifier", xgb.XGBClassifier())
]) 

oblimax_pipeline.fit(X_train, y_train)

eval_data = X_test.copy()
eval_data["ahi"] = y_test

preds = oblimax_pipeline.predict(X_test)
classification_rep = classification_report(y_test, preds)
acc = accuracy_score(y_test, preds)
recall = recall_score(y_test, preds)    

mlflow.autolog(disable=True) #turn it off


2026/04/20 14:56:17 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '009905212db34d17862c409c9a35d7aa', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/04/20 14:56:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Inte